# Programmation Python  pour les mathématiques

**Julien Guillod**, [Sorbonne Université](http://www.sorbonne-universite.fr/),
Licence <a href="https://creativecommons.org/licenses/by-nc-nd/4.0/">CC BY-NC-ND</a>

L'entier des chapitres est disponible au format
[HTML](https://python.guillod.org//) et [PDF](https://python.guillod.org//python.pdf).
Ce notebook peut également être exécuté sur [GESIS](https://notebooks.gesis.org/binder/v2/gh/guillod/python/master?filepath=chap08.ipynb).

https://fr.wikipedia.org/wiki/Th%C3%A9orie_des_graphes

https://medium.com/apprentice-journal/the-graph-theory-an-introduction-in-python-5906d5be0e4b

$$\url{https://www.python.org/doc/essays/graphs/}$$

# 8 Théorie des graphes

<div id="ch:graphes"></div>

Un graphe est un couple $G = (X, E)$ constitué d'un ensemble $X$, non vide et fini, et d'un ensemble $E$ de paires d'éléments de $X$. Les éléments de $X$ sont les sommets du graphe $G$, ceux de $E$ sont les arêtes du graphe $G$.
Un graphe est orienté si les arêtes ont une direction, c'est-à-dire si les couples d'éléments de $E$ sont des listes ordonnées, c'est-à-dire telle que $(i,j) \in E$ n'est pas équivalent à $(j,i)\in E$. Ici seuls des graphes non-orientés, c'est-à-dire dont les paires d'éléments de $X$ sont des ensembles non ordonnés ($\{i,j\} \in E$) sont considérés.

Par exemple, le graphe complet à $n$ sommets $K_n$ est le graphe de sommets $X = {1,2,\dots,n}$ ayant comme arêtes les parties à deux éléments de $X$. En particulier, $K_4 = (X,E)$ où $X=\{1,2,3,4\}$ et $E = \big\{ \{1,2\}, \{1,3\}, \{1,4\}, \{2, 3\}, \{2, 4\}, \{3, 4\}  \big\}$.

**Concepts abordés:**

* graphes non-orientés

* représentation comme dictionnaire

* utilisation des frozensets

* matrice d'adjacence

* recherche de chemins et de triangles

* fonction récursive



In [13]:
import numpy as np
# je me suis permis d'utiliser le module itertools qui fournit les permutations et combinaisons d'un ensemble
# https://docs.python.org/fr/3.9/library/itertools.html
from itertools import permutations, combinations

https://www.geeksforgeeks.org/python-ways-to-remove-duplicates-from-list/

# Exercice 8.1: Graphes comme dictionnaires

Une façon de représenter un graphe $G$ est de le faire avec un dictionnaire dont les clés sont
les sommets, et la valeur associée à chaque clé $x \in X$ est un ensemble contenant les voisins de $x$.

**a)**
Construire sous forme de dictionnaire les graphes suivants:

<center><img src="https://python.guillod.org/fig/graphes.png" style="width:90%;max-width:800px;"></center>




**b)**
Écrire une fonction `complet(n)` qui construit comme dictionnaire le graphe complet $K_n$.



In [19]:
def complet(n):
    K = dict()
    s = set()
    X={i for i in range(1,n+1)}
    l = list(permutations(X,2)) # le graphe complet contient tous les arrangements des éléments de X pris deux à deux
    for c in l:
        if c[0] not in K:
            K[c[0]] = {c[1]}
        else:
            K[c[0]].add(c[1])
    return K
assert complet(4) == {1: {2, 3, 4}, 2: {1, 3, 4}, 3: {1, 2, 4}, 4: {1, 2, 3}}


**c)**
Un graphe donné sous forme de dictionnaire contient l'information plusieurs fois.  Écrire une fonction `corriger(graphe)` permettant de rajouter les éléments manquants de sorte que pour tout sommet `x`, si `y` appartient à `graphe[x]`, alors `y` est aussi une clé et `x` appartient à `graphe[y]`. Tester cette fonction.



In [ ]:
def corriger(graphe):
    # K est le graphe à corriger, K1 le graphe corrigé, s et a sont les sommets et adjacents de K respectivement
    K = graphe
    K1=dict()
    for s in K:
        if s not in K1:
            K1[s] = set()
        for a in K[s]:
            K1[s].add(a)
            if a not in K1:
                K1[a] = set()
                K1[a].add(s)
            else:
                K1[a].add(s)
    return K1
    
assert corriger({1:{3,4,2},3:{2}}) == {1: {2, 3, 4}, 2: {1, 3}, 3: {1, 2}, 4: {1}}


**d)**
Écrire une fonction qui retourne l'ensemble (type `set`) de toutes les arêtes d'un graphe représenté par un dictionnaire.

**Indication:**
Les ensembles sont mutables et donc pas hashables.



In [20]:
def aretes(K):
    c = list()
    for sommet in K:
        for v in K[sommet]:
            if {sommet,v} not in c: # verifie si l'arête exite déjà dans la liste (1,2)=(2,1)
                c.append({sommet,v})
    return {(s,v) for s,v in c} # retourne l'ensemble des arêtes en forme de couple (hashable)

assert aretes(complet(6)) == {(1, 2),
 (1, 3),
 (1, 4),
 (1, 5),
 (1, 6),
 (2, 3),
 (2, 4),
 (2, 5),
 (2, 6),
 (3, 4),
 (3, 5),
 (3, 6),
 (4, 5),
 (4, 6),
 (5, 6)}


**e)**
<span style="color:red">!</span> Écrire une fonction permettant de déterminer si deux sommets sont connectés par un chemin ou non et retourne le chemin si oui.

**Indication:**
Utiliser une fonction récursive.



In [8]:
def chemin(K, i, f, c=list()):
    c = c + [i]
    if i == f:
        return c
    if i not in K:
        return []
    for n in K[i]:
        if n not in c:
            cn = chemin(K, n, f, c)
            return cn
    return None

assert chemin({1: {2, 3, 4}, 2: {1, 3}, 3: {1, 2}, 4: {1}},2,3) == [2, 1, 3]


**f)**
<span style="color:red">!</span> Écrire une fonction qui retourne tous les chemins entre deux sommets (sans les cycles).



In [9]:
def touschemins(K, i, f, c=list()):
        c = c + [i]
        if i == f:
            return [c]
        if i not in K:
            return []
        cs = []
        for n in K[i]:
            if n not in c:
                cs_ = touschemins(K, n, f, c)
                for nc in cs_:
                    if nc not in cs:
                        cs.append(nc)
        return cs

assert touschemins({1: {2, 3, 4}, 2: {1, 3, 4}, 3: {1, 2, 4}, 4: {1, 2, 3}},2,3) == [[2, 1, 3], [2, 1, 4, 3], [2, 3], [2, 4, 1, 3], [2, 4, 3]]

# Exercice 8.2: Triangles dans un graphe

Un triangle dans un graphe est un ensemble de trois sommets reliés entre eux par trois arêtes. La recherche et l'analyse des triangles dans un graphe est importante pour comprendre sa structure.

**a)**
Déterminer mathématiquement le nombre de sous-ensembles de cardinal trois que possède un ensemble de sommets $X$.



In [16]:
def nss(X, n=3):
    return len(list(map(set, combinations(X, n))))

assert nss({0,1,2,3}) == 4


**b)**
Écrire une fonction retournant l'ensemble des triangles d'un graphe.



In [21]:
def triangles(K):
    L=list()
    A = aretes(K)
    s = set(dict.keys(K))
    t = list(map(set, combinations(s, 3))) # tous les triangles possibles de K sont dans t
    for tr in t:
        c = list(map(tuple, combinations(tr, 2))) # si le triangle est présent dans K, toutes les aretes qui le composent doivent être présentes
        add = True
        for couple in c:
            if couple not in A:
                add = False
                break
        if add == True:
            L.append(tr)
    return L
                
assert triangles({1: {2, 3, 4}, 2: {1, 3}, 3: {1, 2,4}, 4: {1,3}}) == [{1, 2, 3}, {1, 3, 4}]


A chaque graphe $G=(X,E)$ correspond une unique matrice $A$ symétrique de taille $n \times n$ avec $n=|X|$ définie par:

$$
A_{ij}=\begin{cases}
1\,, & \text{si}\;\{i,j\}\in E\,,\\ 
0\,, & \text{si}\;\{i,j\}\notin E\,.
\end{cases}
$$

Cette matrice est appelée la matrice d'adjacence du graphe $G$.

**c)**
Définir une fonction retournant la matrice d'adjacence d'un graphe.

**Indication:**
Faire attention que les sommets ne sont pas forcément indexés par des entiers compris entre $0$ et $n$ dans le dictionnaire.




**d)**
Définir une fonction ayant pour argument une matrice d'adjacence et retournant le graphe correspondant sous forme d'un dictionnaire.




**e)**
En utilisant la matrice d'adjacence $A$ et la matrice $B=A^2$, écrire une fonction retournant l'ensemble des triangles d'un graphe.




**f)**
En utilisant la matrice d'adjacence $A$, écrire une fonction calculant le nombre de triangles.

**Indication:**
Interpréter les entrées de la matrice $A^n$.



# Exercice 8.3: <span style="color:red">!!</span> Module NetworkX

De nombreux algorithmes de la théorie des graphes sont implémentés dans le module networkX, voir la documentation [ici](https://networkx.github.io/documentation/).

**a)**
Suivre le tutoriel de networkX disponible [ici](https://networkx.github.io/documentation/stable/tutorial.html).

**b)**
Analyser un des graphes téléchargeables à l'adresse: <https://github.com/gephi/gephi/wiki/Datasets> ou <https://snap.stanford.edu/data/>.

